# D111 — Inside Python Packages and Wheels

A wheel (`.whl`) is a ZIP archive containing an installable Python distribution. Some wheels contain only Python files; others contain native code compiled for a particular operating system, CPU architecture, Python implementation, and Python version.

In this lab we download and extract:

1. NumPy for the current Windows Python — compiled `.pyd` modules and DLLs.
2. `python-dateutil` — platform-independent Python wheels.
3. NumPy for CPython 3.12 on Linux x86-64 — compiled Linux `.so` modules.

The files are stored under the disposable teaching location `C:\tmp\wheel-lab`; they are not installed into the environment.

## 1. Prepare the directories

`pip download` downloads distributions without installing them. We call pip through the notebook kernel's Python.

In [ ]:
from pathlib import Path
import subprocess
import sys
import zipfile

lab = Path(r"C:\tmp\wheel-lab")
windows_wheels = lab / "wheels" / "windows"
pure_wheels = lab / "wheels" / "pure-python"
linux_wheels = lab / "wheels" / "linux-x86_64-cp312"

for folder in (windows_wheels, pure_wheels, linux_wheels):
    folder.mkdir(parents=True, exist_ok=True)

print("Kernel Python:", sys.executable)
print("Lab directory:", lab)

## 2. Windows NumPy: a package containing native code

This downloads the NumPy wheel compatible with the current Windows kernel. `--only-binary=:all:` requires a wheel, and `--no-deps` keeps this example focused on NumPy.

Command Prompt commands (activate the environment first):

```bat
C:\Users\GOPALAKRISHNANSUBRAM\dataeng\Scripts\activate.bat
python -m pip download numpy --only-binary=:all: --no-deps --dest "C:\tmp\wheel-lab\wheels\windows"
```

In [ ]:
subprocess.run([
    sys.executable, "-m", "pip", "download", "numpy",
    "--only-binary=:all:", "--no-deps",
    "--dest", str(windows_wheels),
], check=True)

list(windows_wheels.glob("*.whl"))

## 3. Extract a wheel

Because a wheel is a ZIP archive, Python's `zipfile` can extract it safely and consistently.

From Command Prompt, use Windows `tar`:

```bat
mkdir C:\tmp\wheel-lab\manual-extract
tar -xf C:\tmp\wheel-lab\wheels\windows\numpy-...whl -C C:\tmp\wheel-lab\manual-extract
```

If `tar` is unavailable, PowerShell can extract a copied `.zip`:

```powershell
Copy-Item C:\tmp\wheel-lab\wheels\windows\numpy-...whl C:\tmp\numpy.zip
Expand-Archive C:\tmp\numpy.zip -DestinationPath C:\tmp\wheel-lab\manual-extract
```

In [ ]:
def extract_wheels(wheel_directory, output_directory):
    output_directory.mkdir(parents=True, exist_ok=True)
    extracted = []
    for wheel in wheel_directory.glob("*.whl"):
        destination = output_directory / wheel.stem
        destination.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(wheel) as archive:
            archive.extractall(destination)
        extracted.append(destination)
    return extracted

windows_extracted = extract_wheels(windows_wheels, lab / "extracted" / "windows")
windows_extracted

In [ ]:
def files_with_suffixes(folders, suffixes):
    suffixes = tuple(s.lower() for s in suffixes)
    return [
        file for folder in folders for file in folder.rglob("*")
        if file.is_file() and file.name.lower().endswith(suffixes)
    ]

windows_native = files_with_suffixes(windows_extracted, (".pyd", ".dll"))
print("Windows native files:", len(windows_native))
for file in windows_native[:30]:
    print(file.relative_to(lab))

On Windows, `.pyd` files are Python extension modules implemented as DLLs. Additional `.dll` files may provide bundled numerical runtimes such as BLAS. These binaries were compiled from C, C++, Fortran, or generated native code; a different operating system cannot load them.

## 4. A platform-independent package

`python-dateutil` and its dependency `six` publish universal wheels. Leaving out `--no-deps` downloads the dependency too.

```bat
python -m pip download python-dateutil --only-binary=:all: --dest "C:\tmp\wheel-lab\wheels\pure-python"
```

In [ ]:
subprocess.run([
    sys.executable, "-m", "pip", "download", "python-dateutil",
    "--only-binary=:all:",
    "--dest", str(pure_wheels),
], check=True)

for wheel in pure_wheels.glob("*.whl"):
    print(wheel.name)

In [ ]:
pure_extracted = extract_wheels(pure_wheels, lab / "extracted" / "pure-python")
pure_native = files_with_suffixes(pure_extracted, (".pyd", ".dll", ".so", ".dylib"))

print("Native files:", len(pure_native))
for folder in pure_extracted:
    print("\n", folder.name)
    for file in list(folder.rglob("*.py"))[:12]:
        print(" ", file.relative_to(folder))

Names ending in `py3-none-any.whl` explain the compatibility:

- `py3`: Python 3
- `none`: no specific binary ABI
- `any`: any operating system or CPU architecture

A package can be pure Python while depending on another package. Here both the main wheel and its downloaded dependency are pure Python.

## 5. Download Linux NumPy while running Windows

Pip can download for another target without installing it. All compatibility details are supplied explicitly, and `--no-deps` is required for cross-platform download resolution.

This requests CPython 3.12, Linux x86-64, using the `manylinux2014` compatibility policy:

```bat
python -m pip download numpy --only-binary=:all: --no-deps --platform manylinux2014_x86_64 --implementation cp --python-version 312 --abi cp312 --dest "C:\tmp\wheel-lab\wheels\linux-x86_64-cp312"
```

In [ ]:
subprocess.run([
    sys.executable, "-m", "pip", "download", "numpy",
    "--only-binary=:all:", "--no-deps",
    "--platform", "manylinux2014_x86_64",
    "--implementation", "cp",
    "--python-version", "312",
    "--abi", "cp312",
    "--dest", str(linux_wheels),
], check=True)

for wheel in linux_wheels.glob("*.whl"):
    print(wheel.name)

In [ ]:
linux_extracted = extract_wheels(linux_wheels, lab / "extracted" / "linux-x86_64-cp312")
linux_native = files_with_suffixes(linux_extracted, (".so",))

print("Linux shared-object files:", len(linux_native))
for file in linux_native[:30]:
    print(file.relative_to(lab))

Linux native modules use `.so` (shared object) files rather than Windows `.pyd` and `.dll` files. Their names often include tags such as `cpython-312` and `x86_64-linux-gnu`.

Do **not** add the extracted Linux directory to Windows `PYTHONPATH` and try to import it. Windows cannot load Linux ELF shared objects. Extraction is safe because it only inspects files.

## 6. Optional ARM comparison

Replace the platform with `manylinux2014_aarch64` to obtain the 64-bit ARM Linux wheel:

```bat
python -m pip download numpy --only-binary=:all: --no-deps --platform manylinux2014_aarch64 --implementation cp --python-version 312 --abi cp312 --dest "C:\tmp\wheel-lab\wheels\linux-arm64-cp312"
```

The Linux x86-64 and ARM64 wheels contain code compiled for different CPU instruction sets and are not interchangeable.

## 7. Read wheel metadata

Every wheel includes a `.dist-info` directory. `METADATA` describes the distribution and dependencies; `WHEEL` records wheel compatibility tags.

In [ ]:
all_extracted = windows_extracted + pure_extracted + linux_extracted

for folder in all_extracted:
    wheel_metadata = list(folder.glob("*.dist-info/WHEEL"))
    print(f"\n--- {folder.name} ---")
    if wheel_metadata:
        lines = wheel_metadata[0].read_text(encoding="utf-8").splitlines()
        for line in lines:
            if line.startswith(("Wheel-Version:", "Root-Is-Purelib:", "Tag:")):
                print(line)

### Summary

- Wheels are ZIP archives and can be inspected without installation.
- `py3-none-any` wheels contain platform-independent Python code.
- Windows native wheels may contain `.pyd` and `.dll` files.
- Linux native wheels contain `.so` files.
- Wheel filename tags identify Python, ABI, operating-system, and CPU compatibility.
- Downloading or extracting an incompatible wheel is fine; importing or installing it for the wrong target is not.